In [1]:
from configs import *
from utils import *

import ms_gate_set as mg


import copy

import numpy as np

S0: [1, 2, 3, 4, 9, 10, 11, 12, 13]
output_qubits: ('q1', 'q2', 'q3', 'q4', 'q9', 'q10', 'q11', 'q12', 'q13')
Import new one!!


In [2]:
# 测试函数的常数设置

Is_save = True

filename = "test0"

In [3]:
# 首先我们需要给出 circuit_information_list 的构造方法. 这个会显著影响后续效果的... 
print("当前使用的物理 qubit 是: ",physical_qubits)
print("真机的输出次序是: ",output_qubits)

print("---------------------")

print("CZ layer 的信息是: ",cz_gates_location_list)



当前使用的物理 qubit 是:  ['q1', 'q2', 'q12', 'q13']
真机的输出次序是:  ('q1', 'q2', 'q12', 'q13')
---------------------
CZ layer 的信息是:  [('q1', 'q2', 'q12', 'q13'), ('q1', 'q13', 'q2', 'q12')]


In [4]:
print(basis_list0)

['XX', 'XY', 'XZ', 'YX', 'YY', 'YZ', 'ZX', 'ZY', 'ZZ']


In [5]:
# 下面我们开始计算 circuit_information_list 的内容.
data_np_list = []
config_list_list = []







seed = 80
readout_configuration_list = []
for i in range(2**n_qubits):
    measure_basis=bin(i)[2:].rjust(n_qubits,'0')
    readout_configuration_list.append(measure_basis)

def load_random_config_list2(circuits_information):
    cycle_index = cycle_list.index(int(circuits_information[1]))
    cycle = cycle_list[cycle_index]
    if cycle == 0:
        if n_configs%len(readout_configuration_list)==0:
            config=[]
            for i in range(int(n_configs/len(readout_configuration_list))):
                for j in readout_configuration_list:
                    config.append(tuple([j]))
        else:
            config=generate_random_configurations(n_qubits,n_cycles=cycle,n_configs=n_configs)#,random_state=gmpy2.random_state(seed))
    else:
        config=generate_random_configurations(n_qubits,n_cycles=cycle,n_configs=n_configs)#,random_state=gmpy2.random_state(seed))

    return config

# 下面需要把每一层不同的东西都提前储存起来, 这样后面我们可以直接进行调用

circuits_information_list_list = []
config_list_list = []
basis_list_list = []
basis_list_include_S_H_gate_list = []


for idx, cz_gates_location in enumerate(cz_gates_location_list):
    print("\n" + "-"*30)
    print(f"当前是第 {idx} 层 CZ gate, 位置是: {cz_gates_location}")


    cz_gates_num = len(cz_gates_location) // 2  # 每两个 qubit 一组组成一个 CZ gate
    print(f"cz_gates_num: {cz_gates_num}")

    basis_list = []
    for index in range(len(basis_list0)):
        str0 = ""
        for wndex in range(n_qubits):
            if physical_qubits[wndex] in cz_gates_location:
                if cz_gates_location.index(physical_qubits[wndex]) % 2 == 0:
                    str0 += basis_list0[index][0]  # 对应的 qubit 在 CZ gate 的第一个位置
                else:
                    str0 += basis_list0[index][1]

            else:
                if index % 3 == 0:
                    str0 += 'X'  # 如果这个 qubit 不在 CZ gate 中, 那么我们默认是 X basis
                elif index % 3 == 1:
                    str0 += 'Y'
                else:
                    str0 += 'Z'

        basis_list.append(str0)

    basis_list_list.append(copy.deepcopy(basis_list))

    # 下面定义一个 basis_list 当中包含 是否插入 S 和 H 信息的list 吧
    basis_list_include_S_H_gate = []
    for iindex in range(len(basis_list)):
        if iindex in [0,1,3,4]:
            basis_list_include_S_H_gate.append((basis_list[iindex], True, False))
        elif iindex in [2,5,6,7]:
            basis_list_include_S_H_gate.append((basis_list[iindex], False, False))
            basis_list_include_S_H_gate.append((basis_list[iindex], False, True))
        else:
            basis_list_include_S_H_gate.append((basis_list[iindex], False, False))

    basis_list_include_S_H_gate_list.append(copy.deepcopy(basis_list_include_S_H_gate))


    # circuits_information_list0=[(i,str(j)) for i in basis_list for j in cycle_list] # 这个我们需要加入额外的两个维度决定是否插入 S 和 H gate. 
    # # x = 0 if np.random.random() < (1-(1-noise_Pauli_Probability[index])*noise_level) else 1
    circuits_information_list=[(basis_list_include_S_H_gate[i][0],str(j), basis_list_include_S_H_gate[i][1], basis_list_include_S_H_gate[i][2]) for i in range(len(basis_list_include_S_H_gate)) for j in cycle_list]
    # circuits_information_list = []
    # for iindex in range(len(basis_list)):
    #     for jjndex in cycle_list:
    #         if iindex in [0,1,3,4]:
    #             circuits_information_list.append((basis_list[iindex],str(jjndex), True, False))
    #         elif iindex in [2,5,6,7]:
    #             circuits_information_list.append((basis_list[iindex],str(jjndex), False, False))
    #             circuits_information_list.append((basis_list[iindex],str(jjndex), False, True))
    #         else:
    #             circuits_information_list.append((basis_list[iindex],str(jjndex), False, False))

    
    
    
    
    circuits_information_list_list.append(copy.deepcopy(circuits_information_list))

    print(circuits_information_list)



    print(f"basis_list: {basis_list}")
    print(f"basis_list_H_S: {basis_list_include_S_H_gate}")


    # 我们还是需要把全部的 config 都准备好才可以. 要不然会非常出问题的. 
    config_list = []
    for index in range(len(circuits_information_list)):
        circuits_information = circuits_information_list[index]
        config_list.append(load_random_config_list2(circuits_information))


    config_list_list.append(copy.deepcopy(config_list))




------------------------------
当前是第 0 层 CZ gate, 位置是: ('q1', 'q2', 'q12', 'q13')
cz_gates_num: 2
[('XXXX', '0', True, False), ('XXXX', '1', True, False), ('XXXX', '2', True, False), ('XXXX', '3', True, False), ('XXXX', '4', True, False), ('XXXX', '5', True, False), ('XYXY', '0', True, False), ('XYXY', '1', True, False), ('XYXY', '2', True, False), ('XYXY', '3', True, False), ('XYXY', '4', True, False), ('XYXY', '5', True, False), ('XZXZ', '0', False, False), ('XZXZ', '1', False, False), ('XZXZ', '2', False, False), ('XZXZ', '3', False, False), ('XZXZ', '4', False, False), ('XZXZ', '5', False, False), ('XZXZ', '0', False, True), ('XZXZ', '1', False, True), ('XZXZ', '2', False, True), ('XZXZ', '3', False, True), ('XZXZ', '4', False, True), ('XZXZ', '5', False, True), ('YXYX', '0', True, False), ('YXYX', '1', True, False), ('YXYX', '2', True, False), ('YXYX', '3', True, False), ('YXYX', '4', True, False), ('YXYX', '5', True, False), ('YYYY', '0', True, False), ('YYYY', '1', True, False),

In [6]:
def CZ_Noise_Learning_sim(measure=physical_qubits, stats=1024*32*4, name='CZ_Noise_Learning', oc_tomo=False, save=False, noisy=False, plot=False, correct=True, correct_xtalk_z=None,cz_gates_location = None,circuits_information_list = None,config_list = None):

    
    def func(curr_idx):

        circuits_information = circuits_information_list[curr_idx]
        random_config_list = config_list[curr_idx]
        output_list_list = []
        
        for random_config_tuple in random_config_list:
            
            MQ_Circuit_obj = mg.MQ_Circuit(len(measure))
            mg.construction_cz_gate(MQ_Circuit_obj, circuits_information, random_config_tuple,measure,cz_gates_location,Is_twirling=True)



            # # 在这里 MQ_Circuit_obj 已经完成了所有的构造, 下面我们就可以开始进行电路模拟了. 
            # alg_noise = MQ_Circuit_obj.circuit
            

            # sim = Simulator('mqmatrix', alg_noise.n_qubits)
            # # 这个之前需要 noise adder 才可以的

            # sim.apply_circuit(alg_noise)

            # circ_m = Circuit()
            # for index in range(alg_noise.n_qubits):
            #     circ_m += Measure(f"q{index}").on(index)

            # output = sim.sampling(circ_m, shots=stats)
            output = MQ_Circuit_obj.sampling(shots=stats)
            output_dict = output.data

            output_list = [0]*2**MQ_Circuit_obj.qubits_num

            for key in output_dict:
                reversed_int = int(key[::-1], 2)
                output_list[reversed_int] = output_dict[key]
                # print(key, output_dict[key], reversed_int)

            # 下面需要调换次序并且转换为 vectoer

            output_list_list += output_list

            # # 下面我们开始运行电路就可以了.... 
            # alg[gates.Sync(alg.qubits)]
            # alg[gates.Measure(qubits)]
            # alg.compile(correct_xtalk_z=correct_xtalk_z)
            # reqs.append(run_alg(server, alg, params['stats']))

        return output_list_list
    
    if len(circuits_information_list) == 1:
        return func(0)
    
    else:
        output0_list = []
        for curr_idx in range(len(circuits_information_list)):
            output0 = [curr_idx] + func(curr_idx)
            output0_list.append(output0)
            print(f"正在进行第 {curr_idx+1} 个电路的测量, 共 {len(circuits_information_list)} 个电路.")

        # data = expt.run(func, save=save, noisy=noisy)
        return output0_list


In [7]:
# 下面我们开始学习数据


# 下面对于输出的结果, 需要对于 qubit 的次序进行调整. 


def permute_list_by_tuple(from_tuple, to_tuple, lst):
    """
    将 lst 中的元素顺序从 from_tuple 变换为 to_tuple 所对应的顺序。

    参数：
    - from_tuple: 原始顺序的标签 tuple，例如 ("x1", "x2", "x3")
    - to_tuple: 目标顺序的标签 tuple，例如 ("x2", "x3", "x1")
    - lst: 与 from_tuple 顺序对应的 list，例如 [10, 20, 30]

    返回：
    - permuted_lst: 重新排序后的 list，例如 [20, 30, 10]
    """
    index_map = [from_tuple.index(label) for label in to_tuple]
    # print("index_map",index_map,to_tuple)
    output0 = [lst[i] for i in index_map]
    permuted_str = "".join(str(x) for x in output0)
    return permuted_str


def int_to_bin_str(n, length):
    """
    将整数 n 转换为长度为 length 的二进制字符串，前导补零。

    参数：
    - n: 非负整数，例如 5
    - length: 目标二进制字符串长度，例如 4

    返回：
    - 二进制字符串，例如 "0101"
    """
    return format(n, f'0{length}b')


def bin_str_to_int(bin_str):
    """
    将二进制字符串转换为整数。

    参数：
    - bin_str: 二进制字符串，例如 "0101"

    返回：
    - 整数值，例如 5
    """
    return int(bin_str, 2)


qubit_num = len(output_qubits)

def huanxu_function(data_np):

    new_index_list = []
    for index in range(2**qubit_num):
        qubit_idx = int_to_bin_str(index, qubit_num)
        # print(qubit_idx)
        qubit_idx2 = permute_list_by_tuple(output_qubits, physical_qubits, qubit_idx)
        # print(qubit_idx2)
        # bin_str_to_int(qubit_idx2)
        new_index_list.append(bin_str_to_int(qubit_idx2))

    if data_np.shape[0] == 1:
        # 说明这个时候没有序列编号, 直接进行 换次序就可以了
        data_np_new = np.zeros_like(data_np)
        repeat_num = data_np.shape[1]//(2**qubit_num)

        for index in range(repeat_num):
            for jndex in range(2**qubit_num):
                data_np_new[:,index*2**qubit_num + new_index_list[jndex]] = data_np[:,index*2**qubit_num + jndex]

    else:
        data_np_new = np.zeros_like(data_np)
        repeat_num = (data_np.shape[1]-1)//(2**qubit_num)

        data_np_new[:,0] = data_np[:,0]

        for index in range(repeat_num):
            for jndex in range(2**qubit_num):
                data_np_new[:,index*2**qubit_num + new_index_list[jndex]+1] = data_np[:,index*2**qubit_num + jndex+1]

    return data_np_new




data_np_list = []




for idx, cz_gates_location in enumerate(cz_gates_location_list):


    cz_gates_num = len(cz_gates_location) // 2  # 每两个 qubit 一组组成一个 CZ gate
    print(f"cz_gates_num: {cz_gates_num}")

    basis_list = []
    for index in range(len(basis_list0)):
        str0 = ""
        for wndex in range(n_qubits):
            if physical_qubits[wndex] in cz_gates_location:
                if cz_gates_location.index(physical_qubits[wndex]) % 2 == 0:
                    str0 += basis_list0[index][0]  # 对应的 qubit 在 CZ gate 的第一个位置
                else:
                    str0 += basis_list0[index][1]

            else:
                if index % 3 == 0:
                    str0 += 'X'  # 如果这个 qubit 不在 CZ gate 中, 那么我们默认是 X basis
                elif index % 3 == 1:
                    str0 += 'Y'
                else:
                    str0 += 'Z'

        basis_list.append(str0)

    # basis_list = [wndex * cz_gates_num for wndex in basis_list0] # 这个需要照顾 Qubit 的位置才可以的. 

    # 这个地方我们需要依据 CZ gate 的拓扑结构来定义 basis_list, 确保这个长度和 物理qubit 对应而不是 CZ gate 对应的. 


    # circuits_information_list=[(i,str(j)) for i in basis_list for j in cycle_list]
    circuits_information_list = circuits_information_list_list[idx]
    config_list = config_list_list[idx]











    


    data_output = CZ_Noise_Learning_sim(cz_gates_location = cz_gates_location, circuits_information_list = circuits_information_list,config_list = config_list)
    print("DONE!")
    print(data_output)
    data_np = np.array(data_output)

    data_np = huanxu_function(data_np)

    data_np_list.append(data_np)


    # np.savez(filename + "CZ_layer1.npz",data_np = data_np, config_list = config_list)
    if Is_save:
        # np.savez(filename + f"CZ_layer_list{idx}.npz",data_np_list = data_np_list, config_list = config_list_list, cz_gates_location_list = cz_gates_location_list,physical_qubits = physical_qubits,cycle_list = cycle_list)
        np.savez(
            filename + f"CZ_layer_list{idx}.npz",
            data_np_list=np.array(data_np_list, dtype=object),
            config_list_list=np.array(config_list_list, dtype=object),
            cz_gates_location_list=np.array(cz_gates_location_list, dtype=object),
            physical_qubits=np.array(physical_qubits, dtype=object),
            cycle_list=np.array(cycle_list, dtype=object),
            basis_list_list = np.array(basis_list_list, dtype=object),
            basis_list_include_S_H_gate_list = np.array(basis_list_include_S_H_gate_list, dtype=object),
        )





cz_gates_num: 2
正在进行第 1 个电路的测量, 共 78 个电路.
正在进行第 2 个电路的测量, 共 78 个电路.
正在进行第 3 个电路的测量, 共 78 个电路.
正在进行第 4 个电路的测量, 共 78 个电路.
正在进行第 5 个电路的测量, 共 78 个电路.
正在进行第 6 个电路的测量, 共 78 个电路.
正在进行第 7 个电路的测量, 共 78 个电路.
正在进行第 8 个电路的测量, 共 78 个电路.
正在进行第 9 个电路的测量, 共 78 个电路.
正在进行第 10 个电路的测量, 共 78 个电路.
正在进行第 11 个电路的测量, 共 78 个电路.
正在进行第 12 个电路的测量, 共 78 个电路.
正在进行第 13 个电路的测量, 共 78 个电路.
正在进行第 14 个电路的测量, 共 78 个电路.
正在进行第 15 个电路的测量, 共 78 个电路.
正在进行第 16 个电路的测量, 共 78 个电路.
正在进行第 17 个电路的测量, 共 78 个电路.
正在进行第 18 个电路的测量, 共 78 个电路.
正在进行第 19 个电路的测量, 共 78 个电路.
正在进行第 20 个电路的测量, 共 78 个电路.
正在进行第 21 个电路的测量, 共 78 个电路.
正在进行第 22 个电路的测量, 共 78 个电路.
正在进行第 23 个电路的测量, 共 78 个电路.
正在进行第 24 个电路的测量, 共 78 个电路.
正在进行第 25 个电路的测量, 共 78 个电路.
正在进行第 26 个电路的测量, 共 78 个电路.
正在进行第 27 个电路的测量, 共 78 个电路.
正在进行第 28 个电路的测量, 共 78 个电路.
正在进行第 29 个电路的测量, 共 78 个电路.
正在进行第 30 个电路的测量, 共 78 个电路.
正在进行第 31 个电路的测量, 共 78 个电路.
正在进行第 32 个电路的测量, 共 78 个电路.
正在进行第 33 个电路的测量, 共 78 个电路.
正在进行第 34 个电路的测量, 共 78 个电路.
正在进行第 35 个电路的测量, 共 78 个电路.
正在进行第 36 个电路的测量, 共 78 个电路.
正在进行第 37 个电路的测量, 共 78

In [8]:
if Is_save:
    # np.savez(filename + f"CZ_layer_list{yndex}.npz",data_np_list = data_np_list, config_list = config_list_list, cz_gates_location_list = cz_gates_location_list,physical_qubits = physical_qubits,cycle_list = cycle_list)
    np.savez(
        filename + f"CZ_layer_list.npz",
        data_np_list=np.array(data_np_list, dtype=object),
        config_list_list=np.array(config_list_list, dtype=object),
        cz_gates_location_list=np.array(cz_gates_location_list, dtype=object),
        physical_qubits=np.array(physical_qubits, dtype=object),
        cycle_list=np.array(cycle_list, dtype=object),
        basis_list_list = np.array(basis_list_list, dtype=object),
        basis_list_include_S_H_gate_list = np.array(basis_list_include_S_H_gate_list, dtype=object),
    )